In [ ]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline


# 1. Pipeline-Safe Correlation Filter (Compatible with set_output)
class CorrelationFilter(BaseEstimator, TransformerMixin):

  def __init__(self, threshold=0.95):
    self.threshold = threshold
    self.to_drop_ = []
    self.feature_names_in_ = None

  def fit(self, X, y=None):
    if isinstance(X, pd.DataFrame):
      self.feature_names_in_ = np.array(X.columns, dtype=object)
      df_x = X
    else:
      self.feature_names_in_ = np.array(
          [f'x{i}' for i in range(X.shape[1])], dtype=object
      )
      df_x = pd.DataFrame(X, columns=self.feature_names_in_)

    corr_matrix = df_x.corr().abs()
    upper_tri = corr_matrix.where(
        np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
    )

    self.to_drop_ = [
        col for col in upper_tri.columns if any(upper_tri[col] > self.threshold)
    ]
    return self

  def transform(self, X):
    if isinstance(X, pd.DataFrame):
      return X.drop(columns=self.to_drop_)
    df_x = pd.DataFrame(X, columns=self.feature_names_in_)
    return df_x.drop(columns=self.to_drop_)

  def get_feature_names_out(self, input_features=None):
    if input_features is None:
      input_features = self.feature_names_in_
    return np.array(
        [col for col in input_features if col not in self.to_drop_],
        dtype=object,
    )


# 2. Load Data
df = pd.read_csv('../dataset/full_wo_fft.csv')

X = df.drop(columns=['Activity', 'subject'])
y = df['Activity']
groups = df['subject']

# 3. Group-Aware Train/Test Split (70% Train, 30% Test by Subject)
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=69)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]

# 4. Define Feature Selection Pipeline
feature_pipeline = Pipeline([
    ('variance', VarianceThreshold(threshold=0.01)),
    ('correlation', CorrelationFilter(threshold=0.95)),
])

# Force pipeline transformers to output pandas DataFrames
feature_pipeline.set_output(transform='pandas')

# 5. Fit ONLY on Train Data, Transform Both Train and Test
print('Fitting feature extraction pipeline on training set...')
X_train_filtered = feature_pipeline.fit_transform(X_train)
X_test_filtered = feature_pipeline.transform(X_test)

# 6. Reconstruct DataFrames using ORIGINAL split indices
df_train_filtered = X_train_filtered.copy()
df_train_filtered.index = train_idx
df_train_filtered['Activity'] = df.loc[train_idx, 'Activity']
df_train_filtered['subject'] = df.loc[train_idx, 'subject']
df_train_filtered['split'] = 'train'

df_test_filtered = X_test_filtered.copy()
df_test_filtered.index = test_idx
df_test_filtered['Activity'] = df.loc[test_idx, 'Activity']
df_test_filtered['subject'] = df.loc[test_idx, 'subject']
df_test_filtered['split'] = 'test'

# 7. Concatenate and Sort by Index to Restore Original Row Order
full_df = pd.concat([df_train_filtered, df_test_filtered]).sort_index()

# 8. Export Combined Dataset
output_path = '../dataset/full_v5.csv'
full_df.to_csv(output_path, index=False)

print(
    f'Train set shape: {df_train_filtered.shape[0]} rows,'
    f' {df_train_filtered.shape[1]} columns'
)
print(
    f'Test set shape:  {df_test_filtered.shape[0]} rows,'
    f' {df_test_filtered.shape[1]} columns'
)
print(
    f'Full dataset saved to "{output_path}" ({full_df.shape[0]} rows, original'
    ' row order restored).'
)
print(
    f'Preserved {len(X_train_filtered.columns)} original feature names out of'
    f' {X.shape[1]}.'
)

In [ ]:
import pandas as pd

# 1. Load original and PCA datasets
df_orig = pd.read_csv('../dataset/full_wo_fft.csv')
df_pca = pd.read_csv('../dataset/full_v5.csv')

# 2. Run Verification Checks
checks = {
    'Row Count Match': len(df_orig) == len(df_pca),
    'Activity Alignment': df_orig['Activity'].equals(df_pca['Activity']),
    'Subject Alignment': df_orig['subject'].equals(df_pca['subject']),
}

# 3. Print Results
print('=== INDEX & ORDER VERIFICATION RESULTS ===\n')
all_passed = True

for check_name, passed in checks.items():
  status = 'PASSED' if passed else 'FAILED'
  print(f'[{status}] {check_name}')
  if not passed:
    all_passed = False

print('\n' + '=' * 42)

if all_passed:
  print('SUCCESS: The PCA dataset perfectly matches the original row order!')
else:
  print('WARNING: Row order or index alignment mismatch detected.')
  # Display first mismatched row for debugging if failed
  mismatches = df_orig[df_orig['Activity'] != df_pca['Activity']]
  print(f'First mismatch at index:\n{mismatches.head(1)}')

# 4. Detailed Breakdown Comparison (First 5 Rows)
print('\n--- First 5 Rows Metadata Comparison ---')
comparison = pd.DataFrame({
    'Orig_Activity': df_orig['Activity'].head(5),
    'PCA_Activity': df_pca['Activity'].head(5),
    'Orig_Subject': df_orig['subject'].head(5),
    'PCA_Subject': df_pca['subject'].head(5),
    'Split': df_pca['split'].head(5),
})
print(comparison)